# ELIMINACION DE COLUMNAS CON VALORES REPETIDOS, MODEM(S), ONT_ID, DESCRIPTION Y CAMBIO TITULO COLUMNAS

In [1]:
import pandas as pd #Importar librería pandas
import os #Importar módulo sistema
import json #Importar módulo json
from datetime import datetime #Importar formateo de fechas

# Cargar configuración desde JSON
with open('config_etl_gold.json', 'r') as file: #Abrir archivo JSON
    config = json.load(file) #Cargar datos del JSON

extr_anio = config['etl_params']['anio'] #Definir año a procesar desde JSON
extr_mes = config['etl_params']['mes'] #Definir mes a procesar desde JSON

# Construir el formato AAMM (ej. 2026, mes 2 -> 2602)
str_yymm = datetime(extr_anio, extr_mes, 1).strftime('%y%m') #Formatear fecha

RUTA_SILVER = "datalake/silver/ont-model_sw-version" #Definir ruta entrada silver
RUTA_GOLD = "datalake/gold/clean" #Definir ruta salida gold

if not os.path.exists(RUTA_GOLD): #Verificar existencia directorio
    os.makedirs(RUTA_GOLD) #Crear directorio gold

# Construir nombres de archivo dinámicos con AAMM
FILE_INPUT = f"{RUTA_SILVER}/master_model+sw_silver_{str_yymm}.parquet" #Definir archivo origen dinámico
FILE_OUTPUT_PARQUET = f"{RUTA_GOLD}/dataset_master_gold_{str_yymm}.parquet" #Definir salida parquet dinámica
FILE_OUTPUT_CSV = f"{RUTA_GOLD}/dataset_master_gold_{str_yymm}.csv" #Definir salida csv dinámica

def generar_gold_final(): #Definir función principal
    print(f"Iniciando Generación Capa Gold Final para el periodo {str_yymm}...") #Imprimir inicio proceso
    
    try: #Iniciar bloque manejo de errores
        print("   Cargando Master Silver Integrado...") #Imprimir estado carga
        df = pd.read_parquet(FILE_INPUT) #Cargar parquet integrado
        
        print("   Eliminando columnas innecesarias...") #Imprimir estado limpieza
        cols_eliminar = ['MODEM(S)_y', 'ont_id', 'description'] #Definir columnas descartables
        df = df.drop(columns=[c for c in cols_eliminar if c in df.columns]) #Eliminar columnas

        print("   Estandarizando nombres de columnas a mayúsculas...") #Imprimir estado renombramiento
        diccionario_renombres = { #Iniciar diccionario renombramiento
            'rx_avg': 'RX_AVG', #Renombrar potencia
            'serial_number': 'SERIAL_NUMBER', #Renombrar serial
            'ont_model': 'ONT_MODEL', #Renombrar modelo
            'software_version': 'SOFTWARE_VERSION' #Renombrar software
        } #Cerrar diccionario renombramiento
        df.rename(columns=diccionario_renombres, inplace=True) #Aplicar renombramiento

        print(f"   Guardando dataset final en {RUTA_GOLD}...") #Imprimir estado guardado
        
        #Generar archivo Parquet
        df.to_parquet(FILE_OUTPUT_PARQUET, index=False) #Exportar archivo parquet
        
        #Generar archivo CSV para validación directa
        df.to_csv(FILE_OUTPUT_CSV, index=False, encoding='utf-8-sig') #Exportar archivo csv

        print(f"[EXITO] Capa Gold generada correctamente para {str_yymm}.") #Imprimir mensaje éxito
        print(f"Total Registros: {len(df)}") #Imprimir total registros
        print(f"Total Columnas Finales: {len(df.columns)}") #Imprimir total columnas
        print(f"Columnas: {list(df.columns)}") #Imprimir lista columnas
        
        print("Muestra (Top 5):") #Imprimir cabecera muestra
        #Mostrar muestra con las columnas recién renombradas en mayúsculas
        cols_muestra = ['CONTRATO', 'RX_AVG', 'SERIAL_NUMBER', 'ONT_MODEL', 'SOFTWARE_VERSION'] #Seleccionar columnas muestra
        cols_muestra = [c for c in cols_muestra if c in df.columns] #Validar existencia columnas
        print(df[cols_muestra].head()) #Imprimir registros
        
    except FileNotFoundError as e: #Capturar error archivo no encontrado
        print(f"[ERROR] Archivo Silver integrado no encontrado: {e}") #Imprimir mensaje error
    except Exception as e: #Capturar otros errores
        print(f"[ERROR] Ha ocurrido un error inesperado en la capa Gold: {e}") #Imprimir mensaje error

if __name__ == "__main__": #Verificar ejecución directa
    generar_gold_final() #Ejecutar función principal

W:\RUBEN\python\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
W:\RUBEN\python\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Iniciando Generación Capa Gold Final para el periodo 2602...
   Cargando Master Silver Integrado...
   Eliminando columnas innecesarias...
   Estandarizando nombres de columnas a mayúsculas...
   Guardando dataset final en datalake/gold/clean...
[EXITO] Capa Gold generada correctamente para 2602.
Total Registros: 424
Total Columnas Finales: 20
Columnas: ['CONTRATO', 'FECHA_PEDIDO', 'MOTIVO_PEDIDO', 'PAQUETE_x', 'DETALLE_PEDIDO1', 'DETALLE_PEDIDO2', 'FECHA_CUMPLIMIENTO', 'COD_TECNICO', 'ESTATUS_x', 'ZONA', 'MODEM(S)_x', 'ESTATUS_y', 'PAQUETE_y', 'FECHA_INSTALACION', 'DIAS_ANTIGUEDAD', 'DIAS_ATENCION', 'RX_AVG', 'SERIAL_NUMBER', 'ONT_MODEL', 'SOFTWARE_VERSION']
Muestra (Top 5):
  CONTRATO     RX_AVG     SERIAL_NUMBER     ONT_MODEL SOFTWARE_VERSION
0    42295 -23.339829  58504F4EDD45BAEA      MN-WIFI2           V1.7.1
1    45662 -16.724111  58504F4E1DE0044A       HG102AT           V1.7.1
2    38705 -23.041187  58504F4EDD2BDAAE      BT-213XR        V1.2.16SP
3    39243 -26.010986  48575443